# **0. Install and Import Libraries**

In [7]:
!pip install -U -q \
  "google-genai" \
  langchain \
  langchain_community \
  langchain-core \
  langchain-google-genai \
  langchain_chroma \
  langchain-text-splitters \
  pypdf \
  chromadb \
  langchain-huggingface \
  sentence-transformers

In [8]:
# Import libraries
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from IPython.display import Markdown
from google import genai
import os
import warnings
warnings.filterwarnings("ignore")

In [9]:
df = pd.read_csv("AI Job Market Dataset.csv")
df.head()

,job_id,job_title,company_size,company_industry,country,remote_type,experience_level,years_experience,education_level,skills_python,skills_sql,skills_ml,skills_deep_learning,skills_cloud,salary,job_posting_month,job_posting_year,hiring_urgency,job_openings
0,1,AI Engineer,Startup,Retail,Canada,Remote,Senior,2,Master,0,0,0,1,0,158322,6,2024,Low,4
1,2,Machine Learning Engineer,MNC,Technology,Australia,Hybrid,Mid,0,Bachelor,1,1,1,0,1,163666,11,2026,High,9
2,3,Machine Learning Engineer,MNC,Technology,Germany,Onsite,Mid,14,Master,1,0,1,0,1,158556,3,2026,High,9
3,4,Business Analyst,Startup,Healthcare,Germany,Remote,Mid,9,Master,0,1,0,1,1,95775,3,2025,High,7
4,5,Data Scientist,MNC,Healthcare,Germany,Hybrid,Mid,5,Master,1,1,1,0,0,111873,12,2021,Low,2


# **1. LLM**

## Set up

In [10]:
# Import library
from dotenv import load_dotenv
import os

# Load the `.env` file that contains secret variables like API Keys
load_dotenv()

# We set the Google API Key for authentication when using Google's AI models. This key is necessary to access the services and is kept secret to prevent unauthorized use.
# Ensure you keep the API Key secret in real projects to prevent others from stealing your quota.
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=GEMINI_API_KEY)

In [11]:
# Fetch all available models from the client's model list
all_models = list(client.models.list())

# Simple check to identify if our API Key is problematic or if there is no internet connection
if not all_models:
  print('Warning: No models found. Please ensure the API Key is valid')
else:
  # Embedding model is used to convert our text data (character by character) into a collection of numbers (vectors).
  print('[EMBEDDING MODELS]')

  for m in all_models:
    actions = [a.lower() for a in m.supported_actions]
    if 'embedcontent' in actions or 'embed_content' in actions:
      print(f'ID: {m.name}')

  # Generative Model is the "main brain" function of the chatbot to think of answers and interact in a narrative way.
  print("\n[GENERATIVE MODELS]")

  for m in all_models:
    actions = [a.lower() for a in m.supported_actions]
    if 'generatecontent' in actions or 'generate_content' in actions:
      print(f"ID: {m.name}")

[EMBEDDING MODELS]
ID: models/gemini-embedding-001
ID: models/gemini-embedding-2-preview
ID: models/gemini-embedding-2

[GENERATIVE MODELS]
ID: models/gemini-2.5-flash
ID: models/gemini-2.5-pro
ID: models/gemini-2.0-flash
ID: models/gemini-2.0-flash-001
ID: models/gemini-2.0-flash-lite-001
ID: models/gemini-2.0-flash-lite
ID: models/gemini-2.5-flash-preview-tts
ID: models/gemini-2.5-pro-preview-tts
ID: models/gemma-4-26b-a4b-it
ID: models/gemma-4-31b-it
ID: models/gemini-flash-latest
ID: models/gemini-flash-lite-latest
ID: models/gemini-pro-latest
ID: models/gemini-2.5-flash-lite
ID: models/gemini-2.5-flash-image
ID: models/gemini-3-pro-preview
ID: models/gemini-3-flash-preview
ID: models/gemini-3.1-pro-preview
ID: models/gemini-3.1-pro-preview-customtools
ID: models/gemini-3.1-flash-lite-preview
ID: models/gemini-3.1-flash-lite
ID: models/gemini-3-pro-image-preview
ID: models/gemini-3-pro-image
ID: models/nano-banana-pro-preview
ID: models/gemini-3.1-flash-image-preview
ID: models/gem

In [12]:
EMBEDDING_MODEL = "models/gemini-embedding-2-preview"
CHAT_MODEL = "gemini-3.1-flash-lite-preview"
EMBEDDING_MODEL_HF = "sentence-transformers/all-MiniLM-L6-v2"

In [13]:
chat_model = ChatGoogleGenerativeAI(
  google_api_key=GEMINI_API_KEY,
  model=CHAT_MODEL
)

## Extract Document

In [14]:
# Load and extract PDF
from pdfminer.high_level import extract_text
import os

cv_text = extract_text("CV_Prayoga Rasyid Sudrajat-1-2.pdf")
print(cv_text[:1000])

Prayoga Rasyid Sudrajat 
Aspiring Data Scientist 
Jakarta, Indonesia | +6281281915755 | prayogarasyid26@gmail.com | LinkedIn | Github 

SUMMARY 
Results-oriented  Physics  graduate  from  Universitas  Padjadjaran  with  a  strong  analytical  foundation  and 
hands-on experience in manufacturing environments and data-driven project management. Currently enrolled 
in  Hacktiv8  Data  Science  Bootcamp,  gaining  hands-on  exposure  to  Python,  SQL,  ETL  processes,  data 
visualization,  and  machine  learning  fundamentals.  Proven  track  record  in  leading  large-scale  student 
organizations and executing complex data analytics projects. Passionate about translating raw operational data 
into actionable business insights to drive efficiency and revenue growth. 

EDUCATION 
Hacktiv8 Bootcamp 
Data Science Program. 

Jakarta, Indonesia 
04/2026 - 07/2026 

Padjadjaran University 
Bachelor of Science (Outstanding Student Nominee of Department of Physics Unpad) 

Sumedang, Indonesia 


In [15]:
import re

def clean_text(text):
    # Hilangkan non-breaking space
    text = text.replace("\xa0", " ")

    # Hilangkan trailing whitespace tiap baris
    text = "\n".join(line.rstrip() for line in text.splitlines())

    # Maksimal dua newline berturut-turut
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [16]:
text_cleaned = clean_text(cv_text)
print(text_cleaned)

Prayoga Rasyid Sudrajat
Aspiring Data Scientist
Jakarta, Indonesia | +6281281915755 | prayogarasyid26@gmail.com | LinkedIn | Github

SUMMARY
Results-oriented  Physics  graduate  from  Universitas  Padjadjaran  with  a  strong  analytical  foundation  and
hands-on experience in manufacturing environments and data-driven project management. Currently enrolled
in  Hacktiv8  Data  Science  Bootcamp,  gaining  hands-on  exposure  to  Python,  SQL,  ETL  processes,  data
visualization,  and  machine  learning  fundamentals.  Proven  track  record  in  leading  large-scale  student
organizations and executing complex data analytics projects. Passionate about translating raw operational data
into actionable business insights to drive efficiency and revenue growth.

EDUCATION
Hacktiv8 Bootcamp
Data Science Program.

Jakarta, Indonesia
04/2026 - 07/2026

Padjadjaran University
Bachelor of Science (Outstanding Student Nominee of Department of Physics Unpad)

Sumedang, Indonesia
2021 – 2026

WORK 

In [17]:
# Chunking
import nltk
nltk.download("punkt_tab")
from langchain_text_splitters import NLTKTextSplitter
from langchain_core.documents import Document

def text_split(text):
    text_splitter = NLTKTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separator="\n\n"
    )

    # Convert to document type
    doc = Document(page_content=text_cleaned)
    chunks = text_splitter.split_documents([doc])

    # for i, chunk in enumerate(chunks):
    #     print(f"Length of chunk {i+1} : {len(chunk)} characters")
    #     print(f"Content of chunk {i+1} :\n{chunk}")
    #     print("="*30)
    
    return chunks

chunks = text_split(text_cleaned)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Embedding

Initialize model

In [18]:
# Using Gemini
embedding_model = GoogleGenerativeAIEmbeddings(
    google_api = GEMINI_API_KEY,
    model = EMBEDDING_MODEL
)

# Using Hugging Face (alternative)
embedding_model_hf = HuggingFaceEmbeddings(
  model_name=EMBEDDING_MODEL_HF,
  model_kwargs={'device': 'cpu'}, # Additional options to specify that the model should run on CPU instead of GPU.
  encode_kwargs={'normalize_embeddings': True} # Additional options to specify that the embeddings should be normalized (converted to unit vectors) after encoding.
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6576.15it/s]


In [19]:
print(type(chunks))

<class 'list'>


In [20]:
print(type(chunks[0]))

<class 'langchain_core.documents.base.Document'>


In [21]:
print(len(chunks))

9


Store embeddings

In [22]:
try:
    db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory="./chroma_db"
    )
    print("Stored to vector db using Gemini.")
except:
    db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model_hf,
        persist_directory="./chroma_db"
    )
    print("Stored to vector db using Hugging Face.")

Stored to vector db using Gemini.


In [23]:
# Set up a connection to ChromaDB to retrieve previously saved embeddings
db_connection = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model
)

# Convert Chroma connection into retriever object
retriever = db_connection.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
    )

In [24]:
# Check response with natural language question
check_response = retriever.invoke(
    "What is special about this person?"
)

print(len(check_response))

4


In [25]:
for index in range(0, len(check_response)):
    print(f'\nEmbedding : {index}')
    print('Text : \n')
    print(check_response[index].page_content)
    print('='*50)


Embedding : 0
Text : 

Proven  track  record  in  leading  large-scale  student
organizations and executing complex data analytics projects.

Passionate about translating raw operational data
into actionable business insights to drive efficiency and revenue growth.

EDUCATION
Hacktiv8 Bootcamp
Data Science Program.

Embedding : 1
Text : 

Proven  track  record  in  leading  large-scale  student
organizations and executing complex data analytics projects.

Passionate about translating raw operational data
into actionable business insights to drive efficiency and revenue growth.

EDUCATION
Hacktiv8 Bootcamp
Data Science Program.

Embedding : 2
Text : 

●  Led  a  28+  member  cross-functional  organization,  overseeing  7  functional  divisions  to  ensure  strategic

alignment and execution discipline.

●  Positioned the organization as a Nominee for Most Collaborative Semi-Autonomous Student Organization

at Universitas Padjadjaran.

SKILLS
General Skills: Data Analysis, Machine Learn

## Prompting

In [26]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [39]:
# Create a message template for the system and user messages.
chat_template = ChatPromptTemplate.from_messages(
  [
    # System Message Prompt Template
    SystemMessage(content='''
      Anda adalah AI yang dapat menjawab pertanyaan berdasarkan konteks dan pertanyaan dari pengguna.
      
      Anda adalah AI Career Advisor untuk profesional Data & AI pada tahun 2026.

      Gunakan informasi dari konteks sebagai sumber utama.
      Jika informasi tidak tersedia pada konteks, katakan bahwa informasi tersebut tidak ditemukan dan jangan mengarang jawaban.

      Berikan jawaban yang:
      - Akurat
      - Ringkas tetapi informatif
      - Berorientasi pada pengambilan keputusan karier
      - Sertakan alasan berdasarkan data jika memungkinkan.
      '''),

    # Human Message Prompt Template
    HumanMessagePromptTemplate.from_template('''
      Jawablah pertanyaan-pertanyaan berikut berdasarkan konteksnya.

      konteks: {context}
      pertanyaan: {question}
      jawaban:
      ''')
  ]
)

In [40]:
output_parser = StrOutputParser()

In [29]:
# This function takes a list of documents (retrieved chunks) and formats them into a single string that can be used as context for the chat model to generate an answer. (Concat)

def format_docs(docs):
  return '\n\n'.join(doc.page_content for doc in docs)

# Fetch from retriever and format the retrieved chunks into a single string that can be used as context for the chat model to generate an answer.
formatted_docs = format_docs(chunks)

# Print the formatted results
print('Result : ')
print(formatted_docs)

Result : 
Prayoga Rasyid Sudrajat
Aspiring Data Scientist
Jakarta, Indonesia | +6281281915755 | prayogarasyid26@gmail.com | LinkedIn | Github

SUMMARY
Results-oriented  Physics  graduate  from  Universitas  Padjadjaran  with  a  strong  analytical  foundation  and
hands-on experience in manufacturing environments and data-driven project management.

Currently enrolled
in  Hacktiv8  Data  Science  Bootcamp,  gaining  hands-on  exposure  to  Python,  SQL,  ETL  processes,  data
visualization,  and  machine  learning  fundamentals.

Proven  track  record  in  leading  large-scale  student
organizations and executing complex data analytics projects.

Passionate about translating raw operational data
into actionable business insights to drive efficiency and revenue growth.

EDUCATION
Hacktiv8 Bootcamp
Data Science Program.

EDUCATION
Hacktiv8 Bootcamp
Data Science Program.

Jakarta, Indonesia
04/2026 - 07/2026

Padjadjaran University
Bachelor of Science (Outstanding Student Nominee of Depar

In [32]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    # Input with variables `context` to pass retrieved chunks through `format_docs` and `question` using RunnablePassthrough so it will be used withour modification
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}

    # Pass the input through the `chat_template` to structure the system and user messages for the chat model.
    | chat_template

    # Pass the structured messages through the `chat_model` to generate a response based on the provided context and question.
    | chat_model

    # Lastly, we pass the output from the chat model through the `output_parser` to parse the response into a clean string format that can be easily read and displayed as the final answer to the user's question.
    | output_parser
)

In [41]:
# Query the RAG Chain with a question

response = rag_chain.invoke('''
  Berikan penilaian dari 1-10 kesesuaian kompetensi orang ini dengan keahlian yang harus dimiliki oleh Data Scientist.
  '''
)

In [42]:
Markdown(response)

Berdasarkan konteks yang diberikan, berikut adalah penilaian kesesuaian kompetensi Prayoga Rasyid Sudrajat sebagai Data Scientist:

**Penilaian: 7.5/10**

**Analisis:**

*   **Pendidikan & Fondasi (Skor: 8/10):** Latar belakang pendidikan di bidang Fisika memberikan keunggulan kompetitif yang kuat, karena lulusan Fisika umumnya memiliki kemampuan analitis, pemecahan masalah matematis, dan logika pemrograman yang sangat baik—keterampilan dasar yang krusial bagi seorang Data Scientist.
*   **Keterampilan Teknis (Skor: 7/10):** Saat ini ia sedang menempuh *bootcamp* di Hacktiv8 dan telah terpapar pada *tools* esensial seperti Python, SQL, ETL, visualisasi data, dan machine learning. Penilaian ini belum sempurna karena ia masih dalam tahap "pembelajaran" (*enrolled*), sehingga pengalaman praktisnya secara profesional di bidang data masih perlu pembuktian melalui portofolio proyek nyata.
*   **Pengalaman & *Soft Skills* (Skor: 7/10):** Ia memiliki pengalaman dalam manajemen proyek berbasis data dan kepemimpinan organisasi berskala besar. Hal ini menunjukkan kemampuan untuk menerjemahkan data teknis menjadi *business insights*, yang merupakan nilai tambah tinggi bagi seorang Data Scientist profesional.

**Kesimpulan:**
Prayoga adalah kandidat dengan potensi yang sangat baik (*promising*) karena kombinasi fondasi analitis yang kuat (fisika) dan pelatihan teknis yang relevan. Nilai 7.5 diberikan karena ia sedang dalam fase transisi karier (*aspiring*). Skor akan meningkat menjadi 9 atau 10 setelah ia berhasil menyelesaikan proyek-proyek *end-to-end* yang kompleks dan membuktikan kemampuannya dalam mengimplementasikan model machine learning pada kasus bisnis nyata.

In [30]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size = 500,
#     chunk_overlap = 100,
#     separators = ["\n\n", "\n", ". ", " ", ""],
#     keep_separator=False
# )

# docs = ""
# for page in pdf_file:
#     docs += page

# chunks = text_splitter.split_text(docs)
# print(chunks, "\n")

# for i, chunk in enumerate(chunks):
#   print(f'Length  of chunk {i+1} : {len(chunk)} characters')
#   print('-' * 50)